In [7]:
import torch
import esm
import sys
from rdkit import Chem
from rdkit.Chem import MACCSkeys
from torch_geometric.data import Data, Batch
import math
import pandas as pd
import numpy as np
sys.path.append("../Code/")

from KCM import EitlemKcatPredictor
from KMP import EitlemKmPredictor
from ensemble import ensemble

In [2]:
modelPath = {
    'KCAT':'../Weights/KCAT/iter8_trainR2_0.9408_devR2_0.7459_RMSE_0.7751_MAE_0.4787',
    'KM':'../Weights/KM/iter8_trainR2_0.9303_devR2_0.7163_RMSE_0.6960_MAE_0.4802',
    'KKM':'../Weights/KKM/iter8-trainR2_0.9091_devR2_0.8325_RMSE_0.7417_MAE_0.4896'
}

In [3]:
# Load ESM1v model
model, alphabet = esm.pretrained.esm1v_t33_650M_UR90S_1()
batch_converter = alphabet.get_batch_converter()
#model.eval()

C:\Users\memre\anaconda3\envs\eitlem_env\lib\site-packages\esm\pretrained.py:215: UserWarning: Regression weights not found, predicting contacts will not produce correct results.
  warnings.warn(


In [4]:
def predict(kinetics_type, sequence, smiles):
    # Extratc protein representation
    data = [
    ("protein1", sequence),
    ]
    _, _, batch_tokens = batch_converter(data)
    batch_lens = (batch_tokens != alphabet.padding_idx).sum(1)
    with torch.no_grad():
        results = model(batch_tokens, repr_layers=[33], return_contacts=False)
    token_representations = results["representations"][33]
    sequence_representations = []
    for i, tokens_len in enumerate(batch_lens):
        sequence_representations.append(token_representations[i, 1 : tokens_len - 1])
    # Compute the MACCS Keys of substrate
    mol = Chem.MolFromSmiles(smiles)
    mol_feature = MACCSkeys.GenMACCSKeys(mol).ToList()

    sample = Data(x = torch.FloatTensor(mol_feature).unsqueeze(0), pro_emb=sequence_representations[0])
    input_data = Batch.from_data_list([sample], follow_batch=['pro_emb'])
    if kinetics_type == 'KCAT':
        eitlem = EitlemKcatPredictor(167, 512, 1280, 10, 0.5, 10)
    elif kinetics_type == 'KM':
        eitlem = EitlemKmPredictor(167, 512, 1280, 10, 0.5, 10)
    else:
        eitlem = ensemble(167, 512, 1280, 10, 0.5, 10)
    
    eitlem.load_state_dict(torch.load(modelPath[kinetics_type],map_location=torch.device('cpu')))
    eitlem.eval()
    # Predict kinetics value.
    with torch.no_grad():
        res = eitlem(input_data)
    return math.pow(10,res[0].item())



In [5]:
res = predict("KCAT", 
        "MRAVRLVEIGKPLSLQEIGVPKPKGPQVLIKVEAAGVCHSDVHMRQGRFGNLRIVEDLGVKLPVTLGHEIAGKIEEVGDEVVGYSKGDLVAVNPWQGEGNCYYCRIGEEHLCD\
        SPRWLGINFDGAYAEYVIVPHYKYMYKLRRLNAVEAAPLTCSGITTYRAVRKASLDPTKTLLVVGAGGGLGTMAVQIAKAVSGATIIGVDVREEAVEAAKRAGADYVINASMQD\
        PLAEIRRITESKGVDAVIDLNNSEKTLSVYPKALAKQGKYVMVGLFGADLHYHAPLITLSEIQFVGSLVGNQSDFLGIMRLAEAGKVKPMITKTMKLEEANEAIDNLENFKAIGRQVLIP",
        "COC(=O)C1=CN2CCc3c([nH]c4ccccc34)[C@@]2(C)[C@@H]2CN3CCc4c([nH]c5ccccc45)[C@H]3C[C@H]12")    
print(res) # 1.39

1.3904261610947322


In [11]:
def removeoutlier_col(df,cols):
    Q1 = df[cols].quantile(0.25)
    Q3 = df[cols].quantile(0.75)
    IQR = Q3 - Q1
    df_out = df[~((df[[cols]] < (Q1 - 1.5 * IQR)) |(df[[cols]] > (Q3 + 1.5 * IQR))).any(axis=1)]
    return df_out

df = pd.read_excel('betaGlucosidasewithMutantsOptimumTemperature.xlsx')
output = 'pNP-Glc kcat/Km (1/smM)'
df['Log'+output] = np.log10(df[output])
df_clean = removeoutlier_col(df,'Log' + output).reset_index()
BGL_Label = df_clean[df_clean['Percentage Activity Depending on Optimum Temp']==1]['Log'+output]
sequence_list = df_clean[df_clean['Percentage Activity Depending on Optimum Temp']==1]['Sequence'].values.tolist()

In [12]:
def predict_multiple(kinetics_type, sequences, smiles):
    results_list = []
    for seq in sequences:
        res = predict(kinetics_type, seq, smiles)
        results_list.append(res)
    return results_list

In [14]:
results_bgl = predict_multiple('KKM', sequence_list, 
                               'C1=CC(=CC=C1[N+](=O)[O-])O[C@H]2[C@@H]([C@H]([C@@H]([C@H](O2)CO)O)O)O')

In [16]:
df_results_bgl = pd.DataFrame({'sequences': sequence_list, 'Value':BGL_Label ,
                    'Predicted_label': np.log10(results_bgl)})
df_results_bgl.to_excel('20250827 EITLEM Kinetic_parameters_predicted_label.xlsx')

In [18]:
Predicted_BGL = np.log10(results_bgl)

In [20]:
from sklearn.metrics import r2_score
from sklearn.metrics import mean_absolute_error
from sklearn.metrics import mean_squared_error
from scipy import stats

In [21]:
print('R2 is ' + str(r2_score(BGL_Label, Predicted_BGL)))
print('RMSE is ' + str(mean_squared_error(BGL_Label, Predicted_BGL, squared=False)))
print('MAE is ' + str(mean_absolute_error(BGL_Label, Predicted_BGL)))
print('PCC is ' + str(stats.pearsonr(BGL_Label, Predicted_BGL)[0]))
print('p value is ' + str(stats.pearsonr(BGL_Label, Predicted_BGL)[1]))

R2 is 0.19406256984347292
RMSE is 1.3483092210847634
MAE is 1.0995779921012683
PCC is 0.7348411036287512
p value is 4.6112223792951405e-45
